# DAA 2026/2027 – Ficha Prática 5
**Feature Engineering · Decision Tree Pruning · Hyperparameter Tuning com GridSearchCV**

Dataset: Titanic (`titanic.csv`, formato Kaggle: `PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked`).
Alvo: `Survived` (0 = não sobreviveu, 1 = sobreviveu).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score

RS = 2022   # random_state usado em todo o notebook (reprodutibilidade)
sns.set_style("whitegrid")

## T1 – Carregar o dataset

In [ ]:
titanic = pd.read_csv("titanic.csv")   # ajustar o caminho se necessário
titanic.head()

## T2 – Feature Engineering e Análise Exploratória (EDA)
### 2.1 Visão geral e valores em falta

In [ ]:
titanic.info()
print()
print(titanic.describe(include="all").T)

In [ ]:
# valores em falta (nº e %)
missing = titanic.isnull().sum()
print(pd.DataFrame({"n_null": missing, "%": (missing/len(titanic)*100).round(1)}).query("n_null > 0"))

plt.figure(figsize=(8,3))
sns.heatmap(titanic.isnull(), yticklabels=False, cbar=False, cmap="viridis")
plt.title("Valores em falta"); plt.show()

In [ ]:
# valores únicos por coluna (colunas com 1 só valor ou 1 valor por linha não ajudam)
titanic.nunique()

**Decisões de limpeza**
- `PassengerId`, `Ticket`: identificadores, sem valor preditivo → remover.
- `Cabin`: ~77 % em falta → remover (ou criar `HasCabin`, ver abaixo).
- `Name`: não serve como está, mas contém o **título** (Mr, Mrs, Miss, Master…) → feature nova.
- `Age` (~20 % em falta) → imputar; `Embarked` (2 em falta) → moda.

### 2.2 Criação de novas features

In [ ]:
df = titanic.copy()

# Título extraído do nome e agrupado
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
rare = ["Dr","Rev","Col","Major","Capt","Don","Dona","Jonkheer","Sir","Lady","the Countess"]
df["Title"] = (df["Title"].replace(rare, "Rare")
                          .replace({"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"}))
print(df["Title"].value_counts())

# Tamanho da família e viajar sozinho
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"]    = (df["FamilySize"] == 1).astype(int)

# Tinha cabine registada?
df["HasCabin"] = df["Cabin"].notnull().astype(int)

### 2.3 Tratamento de valores em falta

In [ ]:
# Age: mediana por título (mais informativo do que a mediana global)
df["Age"] = df["Age"].fillna(df.groupby("Title")["Age"].transform("median"))
df["Age"] = df["Age"].fillna(df["Age"].median())

df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Fare"]     = df["Fare"].fillna(df["Fare"].median())

# remover colunas inúteis
df = df.drop(columns=["PassengerId","Name","Ticket","Cabin"], errors="ignore")
print(df.isnull().sum().sum(), "valores em falta restantes")

### 2.4 EDA

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16,9))
sns.countplot(x="Survived", data=df, ax=ax[0,0]);                 ax[0,0].set_title("Distribuição do alvo")
sns.countplot(x="Sex", hue="Survived", data=df, ax=ax[0,1]);      ax[0,1].set_title("Sobrevivência por sexo")
sns.countplot(x="Pclass", hue="Survived", data=df, ax=ax[0,2]);   ax[0,2].set_title("Sobrevivência por classe")
sns.histplot(data=df, x="Age", hue="Survived", bins=30, kde=True, ax=ax[1,0]); ax[1,0].set_title("Idade")
sns.countplot(x="Title", hue="Survived", data=df, ax=ax[1,1]);    ax[1,1].set_title("Título")
sns.countplot(x="FamilySize", hue="Survived", data=df, ax=ax[1,2]); ax[1,2].set_title("Tamanho da família")
plt.tight_layout(); plt.show()

print(df["Survived"].value_counts(normalize=True).round(3))   # balanceamento

### 2.5 Encoding de variáveis categóricas

In [ ]:
# Sex -> binária; Embarked e Title -> one-hot (não têm ordem natural)
df["Sex"] = df["Sex"].map({"male":0, "female":1})
df = pd.get_dummies(df, columns=["Embarked","Title"], drop_first=True, dtype=int)
df.head()

In [ ]:
# Correlações
plt.figure(figsize=(12,9))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=.5)
plt.title("Matriz de correlação (Pearson)"); plt.show()

print(df.corr()["Survived"].sort_values(ascending=False))

> Nota: árvores de decisão **não precisam de scaling/normalização** (ver slides da aula), por isso não escalamos.
> Redundâncias como `FamilySize` vs `SibSp`/`Parch` não prejudicam muito uma árvore, mas podem-se testar removidas.

## T3 – Variáveis de entrada (X) e de saída (y)

In [ ]:
X = df.drop(columns=["Survived"])
y = df["Survived"]
print(X.shape, y.shape)
print(list(X.columns))

## T4 – Divisão treino / teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RS, stratify=y)   # stratify mantém a proporção das classes
print(X_train.shape, X_test.shape)

## T5 – Treinar a Decision Tree (modelo base, sem restrições)

In [ ]:
dt_model = DecisionTreeClassifier(random_state=RS)
dt_model.fit(X_train, y_train)
print("Profundidade:", dt_model.get_depth(), "| Nº de folhas:", dt_model.get_n_leaves())

## T6 – Plot da árvore e avaliação do modelo base

In [ ]:
fig = plt.figure(figsize=(25,12))
plot_tree(dt_model, max_depth=3, feature_names=X.columns, class_names=["Morreu","Sobreviveu"],
          filled=True, rounded=True, fontsize=10)
plt.show()
fig.savefig("dt_plot.png", dpi=150)

In [ ]:
def avalia(modelo, nome):
    """Imprime report, matriz de confusão e devolve métricas para a tabela final."""
    pred = modelo.predict(X_test)
    print(f"===== {nome} =====")
    print(f"Accuracy treino: {modelo.score(X_train, y_train):.3f} | teste: {accuracy_score(y_test, pred):.3f}")
    print(classification_report(y_test, pred, target_names=["Morreu","Sobreviveu"]))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["Morreu","Sobreviveu"])
    plt.title(nome); plt.show()
    rep = classification_report(y_test, pred, output_dict=True)
    return {"Modelo": nome,
            "Acc treino": modelo.score(X_train, y_train),
            "Acc teste": accuracy_score(y_test, pred),
            "F1 macro": rep["macro avg"]["f1-score"],
            "Recall (sobreviveu)": rep["1"]["recall"],
            "Profundidade": modelo.get_depth(),
            "Folhas": modelo.get_n_leaves()}

resultados = [avalia(dt_model, "T6 – Base (sem restrições)")]

**Esperado:** accuracy de treino ≈ 1.00 e de teste bem inferior → **overfitting** (a árvore cresceu até decorar o treino).

## T7 – Tuning com GridSearchCV
Hiperparâmetros: `criterion`, `max_depth`, `min_samples_leaf` (validação cruzada com 5 folds).

In [ ]:
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": list(range(1, 13)),
    "min_samples_leaf": [1, 2, 5, 10, 15, 20, 30],
}
grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=RS), param_grid,
                       cv=5, scoring="accuracy", refit=True, verbose=1, n_jobs=-1)
grid_dt.fit(X_train, y_train)

print("Melhores hiperparâmetros:", grid_dt.best_params_)
print(f"Melhor accuracy CV: {grid_dt.best_score_:.3f}")

In [ ]:
# Top 5 combinações
cv_res = pd.DataFrame(grid_dt.cv_results_)
cols = ["param_criterion","param_max_depth","param_min_samples_leaf","mean_test_score","std_test_score"]
cv_res.sort_values("mean_test_score", ascending=False)[cols].head(5)

## T8 – Avaliação do melhor modelo (GridSearch) e análise crítica

In [ ]:
best_grid = grid_dt.best_estimator_
resultados.append(avalia(best_grid, "T8 – GridSearch"))

fig = plt.figure(figsize=(25,12))
plot_tree(best_grid, feature_names=X.columns, class_names=["Morreu","Sobreviveu"],
          filled=True, rounded=True, fontsize=9)
plt.show(); fig.savefig("dt_plot_grid.png", dpi=150)

# Importância das features
imp = pd.Series(best_grid.feature_importances_, index=X.columns).sort_values()
imp.tail(10).plot.barh(figsize=(7,4), title="Feature importance – GridSearch"); plt.show()

## T9 – Pruning
### 9.1 Pré-pruning: `max_depth`
Testam-se todas as profundidades de 1 até à profundidade da árvore completa (como nos slides).

In [ ]:
max_depth_full = dt_model.get_depth()
gs_depth = GridSearchCV(DecisionTreeClassifier(random_state=RS),
                        {"max_depth": list(range(1, max_depth_full + 1))},
                        cv=5, scoring="accuracy")
gs_depth.fit(X_train, y_train)
print("Melhor max_depth:", gs_depth.best_params_, f"| acc CV = {gs_depth.best_score_:.3f}")
pre_pruned = gs_depth.best_estimator_

# Curva accuracy vs max_depth
d = pd.DataFrame(gs_depth.cv_results_)
tr = [DecisionTreeClassifier(max_depth=k, random_state=RS).fit(X_train, y_train).score(X_train, y_train)
      for k in d["param_max_depth"]]
plt.figure(figsize=(8,4))
plt.plot(d["param_max_depth"], tr, "o-", label="treino")
plt.plot(d["param_max_depth"], d["mean_test_score"], "o-", label="validação (CV)")
plt.xlabel("max_depth"); plt.ylabel("accuracy"); plt.legend(); plt.title("Pré-pruning: max_depth"); plt.show()

### 9.2 Post-pruning: `ccp_alpha` (cost-complexity pruning)

In [ ]:
path = dt_model.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = np.unique(path.ccp_alphas)[:-1]      # o último alpha dá uma árvore só com a raiz
print(len(ccp_alphas), "valores de alpha")

gs_ccp = GridSearchCV(DecisionTreeClassifier(random_state=RS),
                      {"ccp_alpha": ccp_alphas}, cv=5, scoring="accuracy", n_jobs=-1)
gs_ccp.fit(X_train, y_train)
print("Melhor ccp_alpha:", gs_ccp.best_params_, f"| acc CV = {gs_ccp.best_score_:.3f}")
post_pruned = gs_ccp.best_estimator_

In [ ]:
# Accuracy treino/teste vs alpha (como no slide 53)
trees = [DecisionTreeClassifier(random_state=RS, ccp_alpha=a).fit(X_train, y_train) for a in ccp_alphas]
plt.figure(figsize=(8,4))
plt.plot(ccp_alphas, [t.score(X_train, y_train) for t in trees], marker="o", ds="steps-post", label="treino")
plt.plot(ccp_alphas, [t.score(X_test,  y_test)  for t in trees], marker="o", ds="steps-post", label="teste")
plt.axvline(gs_ccp.best_params_["ccp_alpha"], color="g", ls=":", label="alpha escolhido (CV)")
plt.xlabel("ccp_alpha"); plt.ylabel("accuracy"); plt.legend(); plt.title("Accuracy vs alpha"); plt.show()

## T10 – Avaliação dos melhores modelos de cada pruning

In [ ]:
resultados.append(avalia(pre_pruned,  f"T10 – Pré-pruning (max_depth={pre_pruned.max_depth})"))
resultados.append(avalia(post_pruned, f"T10 – Post-pruning (ccp_alpha={post_pruned.ccp_alpha:.5f})"))

for m, nome in [(pre_pruned,"pre"),(post_pruned,"post")]:
    fig = plt.figure(figsize=(25,12))
    plot_tree(m, feature_names=X.columns, class_names=["Morreu","Sobreviveu"], filled=True, rounded=True, fontsize=8)
    plt.show(); fig.savefig(f"dt_plot_{nome}.png", dpi=150)

## T11 – Comparação final e conclusões

In [ ]:
tabela = pd.DataFrame(resultados).set_index("Modelo").round(3)
tabela["Gap (treino-teste)"] = (tabela["Acc treino"] - tabela["Acc teste"]).round(3)
tabela

In [ ]:
# Onde erra cada modelo? (por sexo e classe) – útil para responder a "em que situações performa melhor"
modelos = {"Base": dt_model, "GridSearch": best_grid, "Pré-pruning": pre_pruned, "Post-pruning": post_pruned}
Xt = X_test.copy(); Xt["correto"] = False
linhas = []
for nome, m in modelos.items():
    Xt["correto"] = (m.predict(X_test) == y_test)
    linhas.append(Xt.groupby("Sex")["correto"].mean().rename(lambda s: f"{nome} | {'Mulher' if s==1 else 'Homem'}"))
    linhas.append(Xt.groupby("Pclass")["correto"].mean().rename(lambda c: f"{nome} | Classe {c}"))
print(pd.concat(linhas).round(3).to_string())

### Guia para escrever as conclusões (preencher com os **teus** números)

**T6 – modelo base:** accuracy treino ≈ 1.0 vs teste ≈ ____ → overfitting; árvore com ____ folhas e profundidade ____ (demasiado complexa, instável e difícil de interpretar).

**T8 – GridSearch:** melhores parâmetros `criterion=____, max_depth=____, min_samples_leaf=____`; o gap treino–teste diminuiu para ____; accuracy de teste subiu de ____ para ____. `min_samples_leaf` > 1 e profundidade pequena funcionam como regularização.

**T10 – Pruning:** pré-pruning (`max_depth=____`) vs post-pruning (`ccp_alpha=____`): comparar accuracy, F1 macro, nº de folhas. O pré-pruning é mais simples/rápido (para o crescimento cedo); o post-pruning deixa a árvore crescer e corta depois, podendo encontrar uma estrutura melhor mas é mais custoso.

**Em que situações o modelo se sai melhor?** (ver tabela por sexo/classe) – tipicamente prevê melhor a classe maioritária (não sobreviventes) e mulheres de 1.ª/2.ª classe / homens de 3.ª classe; erra mais em casos “atípicos” (homens de 1.ª classe que sobreviveram, mulheres de 3.ª que morreram).

**Melhor modelo:** o que tiver melhor accuracy/F1 de teste **com menor gap** e menos folhas (princípio da parcimónia), com os parâmetros ____.

**O que melhorar:** mais feature engineering (ex.: `Fare` por pessoa, deck da cabine, grupos de bilhete), tratar desbalanceamento (`class_weight="balanced"`), usar CV repetida em vez de um único split, `min_samples_split`, `max_features`, e métodos ensemble (Random Forest, Gradient Boosting), que reduzem a instabilidade das árvores individuais.